# LahjaLens - Etape 08 : Silver -> E2 -> Gold

Ce notebook ne fait aucun entrainement. Il charge le PipelineModel E2 depuis HDFS, prepare un lot technique de 1 000 commentaires Hespress en Silver, applique l'inference Spark et ecrit Gold.

Les commentaires Hespress ne sont pas labels : aucune accuracy ni Macro-F1 n'est calculee. Les statistiques produites sont descriptives des predictions.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter
import hashlib, json, os, time
from pyspark.sql import SparkSession, functions as F, types as T
from pyspark.ml import PipelineModel

ROOT=Path('/opt/lahjalens'); DATA=ROOT/'data'
INPUT_SAMPLE=DATA/'processed/hespress/comments_sample.csv'
MODEL_RUN_ID='20261115T190123Z_9cb0026f'
MODEL_PATH='hdfs://namenode:9000/data/lahjalens/models/sentiment/experiments/mac_multidataset/'+MODEL_RUN_ID+'/e2'
MODEL_VERSION='e2-afrisenti-mac-'+MODEL_RUN_ID
PREPROCESSING_VERSION='shared_clean_v1_nb05'
LABELS=['negative','neutral','positive']
RUN_ID=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')+'_'+hashlib.sha256(str(time.time_ns()).encode()).hexdigest()[:8]
LOCAL_OUT=DATA/'processed/hespress/e2_inference'/RUN_ID
for child in ('audit','analytics','tests'): (LOCAL_OUT/child).mkdir(parents=True,exist_ok=False)
SILVER_PATH='hdfs://namenode:9000/data/lahjalens/silver/hespress/e2_inference/'+RUN_ID
GOLD_PATH='hdfs://namenode:9000/data/lahjalens/gold/hespress/e2_inference/'+RUN_ID
PREDICTED_AT=datetime.now(timezone.utc).isoformat().replace('+00:00','Z')
spark=(SparkSession.builder.appName('LahjaLens-E2-Inference-'+RUN_ID).master('spark://spark-master:7077')
 .config('spark.hadoop.dfs.replication','2').config('spark.sql.shuffle.partitions','8')
 .config('spark.default.parallelism','4').getOrCreate())
spark.sparkContext.setLogLevel('WARN')
assert spark.sparkContext.master=='spark://spark-master:7077'
spark.sparkContext._jsc.hadoopConfiguration().set('dfs.replication','2')
CLUSTER={'master':spark.sparkContext.master,'application_id':spark.sparkContext.applicationId,'spark_version':spark.version,'python':__import__('sys').version.split()[0],'replication':spark.sparkContext._jsc.hadoopConfiguration().get('dfs.replication')}
def hdfs_exists(uri):
    path=spark.sparkContext._jvm.org.apache.hadoop.fs.Path(uri); fs=path.getFileSystem(spark.sparkContext._jsc.hadoopConfiguration()); return bool(fs.exists(path))
def jwrite(rel,value):
    p=LOCAL_OUT/rel; p.parent.mkdir(parents=True,exist_ok=True); p.write_text(json.dumps(value,ensure_ascii=False,indent=2,allow_nan=False)+'\n',encoding='utf-8')
def twrite(rel,value):
    p=LOCAL_OUT/rel; p.parent.mkdir(parents=True,exist_ok=True); p.write_text(value,encoding='utf-8')
def probability_array(vector):
    return None if vector is None else [float(x) for x in vector.toArray()]
probability_array_udf=F.udf(probability_array,T.ArrayType(T.DoubleType()))
print('RUN_ID',RUN_ID); print('CLUSTER',CLUSTER); print('MODEL_PATH',MODEL_PATH); print('SILVER_PATH',SILVER_PATH); print('GOLD_PATH',GOLD_PATH)

In [ ]:
# Audit des contrats locaux, du modele et des chemins HDFS
assert INPUT_SAMPLE.exists(), f'Input absent: {INPUT_SAMPLE}'
assert not hdfs_exists(SILVER_PATH), 'Refus: le chemin Silver existe deja'
assert not hdfs_exists(GOLD_PATH), 'Refus: le chemin Gold existe deja'
source_schema=T.StructType([
 T.StructField('sample_id',T.StringType(),False),T.StructField('source',T.StringType(),False),T.StructField('source_file',T.StringType(),True),T.StructField('source_record',T.StringType(),True),
 T.StructField('article_id',T.StringType(),True),T.StructField('comment_text',T.StringType(),False),T.StructField('article_topic',T.StringType(),True),T.StructField('article_title',T.StringType(),True),T.StructField('article_date_raw',T.StringType(),True),
 T.StructField('event_time',T.StringType(),True),T.StructField('language',T.StringType(),True),T.StructField('sentiment',T.StringType(),True),T.StructField('theme_label',T.StringType(),True),T.StructField('sentiment_annotator',T.StringType(),True),T.StructField('theme_annotator',T.StringType(),True)])
source=spark.read.schema(source_schema).options(header=True,enforceSchema=False,mode='FAILFAST',encoding='UTF-8',quote='\"',escape='\"',multiLine=True).csv(str(INPUT_SAMPLE)).cache()
input_count=source.count(); assert input_count==1000, f'Expected 1000 comments, got {input_count}'
assert source.where(F.col('comment_text').isNull()|(F.length(F.trim('comment_text'))<=0)).count()==0
assert source.where(F.col('sample_id').isNull()).count()==0
assert source.groupBy('sample_id').count().where(F.col('count')>1).count()==0
assert hdfs_exists(MODEL_PATH), f'Model E2 absent: {MODEL_PATH}'
model=PipelineModel.load(MODEL_PATH)
stage_names=[type(stage).__name__ for stage in model.stages]
assert stage_names[-1]=='LogisticRegressionModel', stage_names
assert getattr(model.stages[-1],'numClasses',None)==3
MODEL_AUDIT={'model_path':MODEL_PATH,'model_version':MODEL_VERSION,'preprocessing_version':PREPROCESSING_VERSION,'pipeline_stages':stage_names,'expected_label_order':LABELS,'input_count':input_count,'source_sample':str(INPUT_SAMPLE),'cluster':CLUSTER}
jwrite('audit/model_audit.json',MODEL_AUDIT)
print('MODEL_LOADED',json.dumps(MODEL_AUDIT,ensure_ascii=False))

In [ ]:
# Materialisation Silver : le texte original est conserve, les champs de prediction sont absents
silver=(source.select(
 F.col('sample_id').alias('event_id'),F.col('comment_text'),F.col('source'),F.col('source_file'),F.col('source_record'),F.col('article_id'),F.col('article_title'),F.col('article_topic'),F.col('article_date_raw'),
 F.when(F.length(F.trim('event_time'))==0,F.lit(None).cast('timestamp')).otherwise(F.to_timestamp('event_time')).alias('event_time'),F.col('language'),
 F.lit(RUN_ID).alias('silver_run_id'),F.lit('hespress_sample_v1').alias('source_version'),F.lit('unlabeled').alias('label_status'),F.lit(PREPROCESSING_VERSION).alias('preprocessing_version'))
 .cache())
silver_count=silver.count(); silver_ids={r['event_id'] for r in silver.select('event_id').collect()}
assert silver_count==input_count and len(silver_ids)==input_count
assert set(silver.columns)=={'event_id','comment_text','source','source_file','source_record','article_id','article_title','article_topic','article_date_raw','event_time','language','silver_run_id','source_version','label_status','preprocessing_version'}
silver.write.mode('errorifexists').parquet(SILVER_PATH)
silver_check=spark.read.parquet(SILVER_PATH); assert silver_check.count()==silver_count
print('SILVER_WRITTEN',silver_count,SILVER_PATH)

In [ ]:
# Smoke test puis inference complete depuis Silver HDFS
silver=spark.read.parquet(SILVER_PATH).cache(); assert silver.count()==1000
smoke=model.transform(silver.limit(5)).select('event_id','comment_text','prediction','probability').collect()
assert len(smoke)==5 and all(int(row['prediction']) in (0,1,2) for row in smoke)
print('SMOKE_OK',[(row['event_id'],int(row['prediction'])) for row in smoke])
predicted=model.transform(silver).withColumn('_probability_array',probability_array_udf('probability')).cache()
prediction_count=predicted.count(); assert prediction_count==silver_count
prob_check=(predicted.withColumn('_prob_sum',F.aggregate('_probability_array',F.lit(0.0),lambda acc,x:acc+x))
 .withColumn('_prediction_valid',F.col('prediction').cast('int').isin(0,1,2)))
assert prob_check.where((F.col('_prob_sum')<0.999999)|(F.col('_prob_sum')>1.000001)).count()==0
assert prob_check.where(~F.col('_prediction_valid')).count()==0
gold=(predicted.select(
 F.col('event_id'),F.col('comment_text'),F.col('source'),F.col('source_file'),F.col('source_record'),F.col('article_id'),F.col('article_title'),F.col('article_topic'),F.col('article_date_raw'),F.col('event_time'),F.col('language'),F.col('silver_run_id'),F.col('source_version'),F.col('label_status'),
 F.col('prediction').cast('int').alias('prediction'),F.when(F.col('prediction')==0,'negative').when(F.col('prediction')==1,'neutral').when(F.col('prediction')==2,'positive').otherwise(None).alias('sentiment'),
 F.lit(MODEL_VERSION).alias('model_version'),F.lit(PREPROCESSING_VERSION).alias('preprocessing_version'),F.to_timestamp(F.lit(PREDICTED_AT)).alias('predicted_at'),
 F.col('_probability_array')[0].cast('double').alias('prob_negative'),F.col('_probability_array')[1].cast('double').alias('prob_neutral'),F.col('_probability_array')[2].cast('double').alias('prob_positive'),
 F.lit(False).alias('probabilities_calibrated'),F.lit(','.join(LABELS)).alias('model_label_order'),F.lit(RUN_ID).alias('inference_run_id'),F.lit(SILVER_PATH).alias('silver_path'),F.lit(GOLD_PATH).alias('gold_path')))
gold_count=gold.count(); gold_ids={r['event_id'] for r in gold.select('event_id').collect()}
assert gold_count==silver_count and gold_ids==silver_ids
assert gold.where(F.col('sentiment').isNull()).count()==0
gold.write.mode('errorifexists').parquet(GOLD_PATH)
gold_check=spark.read.parquet(GOLD_PATH).cache(); assert gold_check.count()==gold_count
jwrite('tests/inference_checks.json',{'input_count':input_count,'silver_count':silver_count,'gold_count':gold_count,'unique_input_ids':len(silver_ids),'unique_gold_ids':len(gold_ids),'prediction_values':sorted({int(r['prediction']) for r in gold.select('prediction').distinct().collect()}),'probabilities_calibrated':False,'model_label_order':LABELS})
print('GOLD_WRITTEN',gold_count,GOLD_PATH)

In [ ]:
# Reproductibilite : meme modele, meme Silver, deux executions de prediction
first=gold.select('event_id','prediction','prob_negative','prob_neutral','prob_positive').alias('first')
second=(model.transform(silver).withColumn('_p',probability_array_udf('probability')).select('event_id',F.col('prediction').cast('int').alias('prediction'),F.col('_p')[0].alias('prob_negative'),F.col('_p')[1].alias('prob_neutral'),F.col('_p')[2].alias('prob_positive')).alias('second'))
repro=(first.join(second,'event_id').select(F.abs(F.col('first.prediction')-F.col('second.prediction')).alias('prediction_delta'),F.abs(F.col('first.prob_negative')-F.col('second.prob_negative')).alias('prob_delta')))
repro_mismatch=repro.where((F.col('prediction_delta')!=0)|(F.col('prob_delta')>1e-12)).count(); assert repro_mismatch==0
jwrite('tests/reproducibility.json',{'mismatched_rows':repro_mismatch,'tolerance':1e-12,'status':'ok' if repro_mismatch==0 else 'failed'})
print('REPRODUCIBILITY_OK',repro_mismatch)

In [ ]:
# Analytique Spark SQL : descriptif des predictions, sans tendance temporelle
gold_check.createOrReplaceTempView('gold_predictions')
overview=spark.sql("SELECT COUNT(*) AS rows, COUNT(DISTINCT event_id) AS unique_event_ids, SUM(CASE WHEN comment_text IS NULL OR TRIM(comment_text)='' THEN 1 ELSE 0 END) AS missing_comment_text, SUM(CASE WHEN sentiment IS NULL THEN 1 ELSE 0 END) AS missing_sentiment FROM gold_predictions").first().asDict()
sentiment_distribution=[r.asDict() for r in spark.sql("SELECT sentiment, COUNT(*) AS rows FROM gold_predictions GROUP BY sentiment ORDER BY sentiment").collect()]
topic_distribution=[r.asDict() for r in spark.sql("SELECT article_topic, COUNT(*) AS rows FROM gold_predictions GROUP BY article_topic ORDER BY article_topic").collect()]
source_distribution=[r.asDict() for r in spark.sql("SELECT source, COUNT(*) AS rows FROM gold_predictions GROUP BY source ORDER BY source").collect()]
topic_sentiment=[r.asDict() for r in spark.sql("SELECT article_topic, sentiment, COUNT(*) AS rows FROM gold_predictions GROUP BY article_topic, sentiment ORDER BY article_topic, sentiment").collect()]
anomalies=[r.asDict() for r in spark.sql("SELECT 'missing_article_id' AS anomaly, COUNT(*) AS rows FROM gold_predictions WHERE article_id IS NULL OR TRIM(article_id)='' UNION ALL SELECT 'invalid_probability' AS anomaly, COUNT(*) AS rows FROM gold_predictions WHERE prob_negative<0 OR prob_neutral<0 OR prob_positive<0 OR prob_negative>1 OR prob_neutral>1 OR prob_positive>1 UNION ALL SELECT 'artificial_temporal_input' AS anomaly, COUNT(*) AS rows FROM gold_predictions WHERE event_time IS NOT NULL").collect()]
assert overview['rows']==1000 and overview['unique_event_ids']==1000 and overview['missing_comment_text']==0 and overview['missing_sentiment']==0
jwrite('analytics/overview.json',overview); jwrite('analytics/sentiment_distribution.json',sentiment_distribution); jwrite('analytics/topic_distribution.json',topic_distribution); jwrite('analytics/source_distribution.json',source_distribution); jwrite('analytics/topic_sentiment.json',topic_sentiment); jwrite('analytics/anomalies.json',anomalies)
ANALYTICS={'overview':overview,'sentiment_distribution':sentiment_distribution,'topic_distribution':topic_distribution,'source_distribution':source_distribution,'topic_sentiment':topic_sentiment,'anomalies':anomalies}
print('ANALYTICS',json.dumps(ANALYTICS,ensure_ascii=False))

In [ ]:
# Manifest et rapport d'execution
manifest={'status':'completed','run_id':RUN_ID,'cluster':CLUSTER,'input_sample':str(INPUT_SAMPLE),'input_count':input_count,'model_path':MODEL_PATH,'model_version':MODEL_VERSION,'preprocessing_version':PREPROCESSING_VERSION,'model_pipeline_stages':stage_names,'label_order':LABELS,'silver_path':SILVER_PATH,'gold_path':GOLD_PATH,'silver_count':silver_count,'gold_count':gold_count,'reproducibility_mismatches':repro_mismatch,'analytics':ANALYTICS,'predicted_at':PREDICTED_AT,'labels_available':False}
jwrite('run_manifest.json',manifest)
report=['# LahjaLens - Integration Silver -> E2 -> Gold','',f'Run : {RUN_ID}',f'Application Spark : {CLUSTER["application_id"]}','', '## Architecture executee','',f'- Silver HDFS : `{SILVER_PATH}`',f'- Modele E2 HDFS : `{MODEL_PATH}`',f'- Gold HDFS : `{GOLD_PATH}`',f'- Cluster : `{CLUSTER["master"]}` avec replication HDFS {CLUSTER["replication"]}.','', '## Controles','',f'- Entrees Hespress : {input_count}',f'- Lignes Silver : {silver_count}',f'- Lignes Gold : {gold_count}',f'- IDs uniques et traces : {len(gold_ids)}',f'- Prediction values : {sorted({int(r["prediction"]) for r in gold.select("prediction").distinct().collect()})}',f'- Reproductibilite : {repro_mismatch} divergence',f'- Probabilites : ordre `{",".join(LABELS)}`, non calibrees', '', '## Limites','', '- Hespress n a pas de verite terrain dans cet echantillon.', '- Les statistiques sont descriptives des sorties du modele et ne sont pas des scores de classification.', '- article_topic est la categorie de l article, pas un theme NLP predit dans le commentaire.', '- event_time est absent dans la source ; aucune tendance temporelle n est calculee.', '- Les sorties Gold sont versionnees et aucune ancienne sortie n est ecrasee.']
twrite('integration_report.md','\n'.join(report)+'\n')
print('COMPLETED',RUN_ID); print('REPORT',LOCAL_OUT/'integration_report.md')